### **Submissão 1A — Modelo NumPy (Implementação Própria)**

**Grupo 1 · MIA · Aprendizagem Profunda**

Modelo: DNN from scratch (NumPy) com Dropout + L2  
Output: `subm1-g1-MIA-A.csv`

In [1]:
import numpy as np
import pandas as pd
import pickle
import sys, os

sys.path.append(os.path.abspath('../src'))
from neuralnet import NeuralNetwork
from layers import DenseLayer, DropoutLayer
from activations import ReLUActivation, SoftmaxActivation
from losses import CategoricalCrossEntropy
from optimizer import Adam
from utils import transform_new_texts

print("1. A carregar modelo NumPy...")
with open('../models/numpy.pkl', 'rb') as f:
    data = pickle.load(f)

transformers = data['transformers']
class_names = data['class_names']
pesos_camadas = data['pesos_camadas']

print(f"   Modelo: {data['modelo_tipo']}")
print(f"   Classes: {list(class_names)}")

1. A carregar modelo NumPy...
   Modelo: DNN D+L2
   Classes: [np.str_('Anthropic'), np.str_('Google'), np.str_('Human'), np.str_('Meta'), np.str_('OpenAI')]


In [2]:
print("2. A carregar dataset de submissão...")
df = pd.read_csv('../data/subm1.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()

textos = df['text'].tolist()
ids = df['id'].tolist()
print(f"   {len(textos)} textos carregados")

2. A carregar dataset de submissão...
   150 textos carregados


In [3]:
print("3. A extrair features...")
X = transform_new_texts(textos, transformers)
print(f"   {X.shape[0]} textos → {X.shape[1]} features")

3. A extrair features...
   150 textos → 2013 features


In [4]:
print("4. A classificar...")

# Reconstruir DNN a partir dos pesos guardados
input_dim = X.shape[1]
output_dim = len(class_names)
adam = Adam(learning_rate=0.0005)
nn_model = NeuralNetwork(loss_func=CategoricalCrossEntropy())

dense_layers = [(i, p) for i, p in enumerate(pesos_camadas) if p is not None]

prev_dim = input_dim
for idx, (pos, (w, b)) in enumerate(dense_layers):
    n_units = w.shape[1]
    layer = DenseLayer(prev_dim, n_units, optimizer=adam)
    layer.weights = w
    layer.biases = b
    nn_model.add(layer)
    prev_dim = n_units
    
    if idx == len(dense_layers) - 1:
        nn_model.add(SoftmaxActivation())
    else:
        nn_model.add(ReLUActivation())

nn_model.set_training(False)
y_pred = np.argmax(nn_model.predict(X), axis=1)
labels_pred = [class_names[i] for i in y_pred]

print(f"   ✅ {len(labels_pred)} previsões feitas")

4. A classificar...
   ✅ 150 previsões feitas


In [ ]:
print("5. A exportar CSV...")

df_out = pd.DataFrame({
    'ID': ids,
    'Text': textos,
    'Labels': labels_pred
})

output_path = '../subm1/subm1-g1-MIA-A.csv'
os.makedirs('../subm1', exist_ok=True)
df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f"✅ Ficheiro guardado: {output_path}")
print(f"\nDistribuição das previsões:")
print(df_out['Labels'].value_counts().to_string())
print(f"\nPrimeiras 10 linhas:")
print(df_out.head(10).to_string(index=False))

5. A exportar CSV...
✅ Ficheiro guardado: ../Subm1/subm1-g1-MIA-A.csv

Distribuição das previsões:
Labels
Human        65
Anthropic    29
Google       25
Meta         19
OpenAI       12

Primeiras 10 linhas:
   ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   